In [13]:
from pathlib import Path
from collections import Counter
import marisa_trie
from symspellpy import SymSpell, Verbosity

DOMAIN_DICTIONARY_PATH = Path("domain_dictionary.txt") 

word_freq = Counter()
with open(DOMAIN_DICTIONARY_PATH, encoding="utf-8") as f:
    for line in f:
        term, _, count = line.rpartition(" ")
        if term:
            word_freq[term] = int(count)

dictionary = set(word_freq)
trie = marisa_trie.Trie(dictionary)

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary(DOMAIN_DICTIONARY_PATH, 0, 1, separator=' ', encoding='utf-8')

DISTANCE_PENALTY = 10

def correct_word_prefix(word, max_edit_distance=2):
    if word in dictionary:
        return word

    if len(word) >= 2:
        completions = trie.keys(word)
        if completions:
            return max(completions, key=lambda w: word_freq.get(w, 0))

    candidates = sym_spell.lookup(word, Verbosity.ALL, max_edit_distance=max_edit_distance)
    if candidates:
        best = max(candidates, key=lambda s: s.count / (DISTANCE_PENALTY ** s.distance))
        return best.term

    segmentation = sym_spell.word_segmentation(word, max_edit_distance=max_edit_distance)
    parts = segmentation.corrected_string.split()
    if (
        len(parts) > 1
        and all(p in dictionary for p in parts)
        and segmentation.distance_sum <= max_edit_distance
        and all(word_freq.get(p, 0) >= 3 for p in parts)
    ):
        return segmentation.corrected_string

    return word

def correct_query_prefix(query, max_edit_distance=2):
    query = query.strip()
    words = query.split()
    return ' '.join(correct_word_prefix(w, max_edit_distance) for w in words)


FileNotFoundError: [Errno 2] No such file or directory: 'domain_dictionary.txt'

In [12]:
print(correct_query_prefix('крсовки'))
print(correct_query_prefix('днисисмо'))
print(correct_query_prefix('кокт'))
print(correct_query_prefix('укропбатон'))

кроссовки
даниссимо
коктейль
укроп батон
